# Week 7 · Notebook 2  Knowledge-Graph Lab

**A carrier → lane → port graph in `networkx` (zero heavy deps): route queries, degree centrality, and an optional Neo4j/Cypher twin.**

```
# Requirements: pip install networkx
```

The optional final section needs Neo4j (AuraDB free or Docker); if it is unavailable the notebook still completes. Part of AI Engineering Lab · ZoroLogistics case study.

## When graphs beat vectors

Vectors capture *similarity*; graphs capture *relationship*. When the question is "how are these things connected, and what can I infer along the path"  multi-hop reachability, cheapest-route-with-a-constraint, exact neighbors  a graph answers it directly, where chunk retrieval only approximates it by luck.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root (assumes cwd == week-NN/)
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro").is_dir():
        sys.path.insert(0, str(_p)); break

from zoro import data
import numpy as np, pandas as pd
import networkx as nx

carriers = data.carriers(20, seed=7)
lanes = data.lanes(20, seed=11)
print("carriers:", len(carriers), "| lanes:", len(lanes))
print("ports seen in lanes:", sorted(lanes["port_region"].unique()))

## Build the graph

Nodes: `carrier`, `lane`, `port`. Edges: `carrier → lane` (`SERVES`, with `cost` = base rate × distance and `days`), and `lane → port` (`ARRIVES_AT`). Carrier→lane assignment is seeded, so the graph is reproducible. NaN distances (planted by the generator) are filled with a default so the graph stays connected.

In [ ]:
G = nx.DiGraph()

ports = sorted(set(data.PORTS) | set(lanes["port_region"].dropna().unique()))
for p in ports:
    G.add_node(p, kind="port")

for _, row in lanes.iterrows():
    distance = row["distance_km"] if not pd.isna(row["distance_km"]) else 900.0
    days = row["avg_transit_days"] if not pd.isna(row["avg_transit_days"]) else 3.0
    G.add_node(row["lane_id"], kind="lane", origin=row["origin"], destination=row["destination"],
               distance_km=float(distance))
    G.add_edge(row["lane_id"], row["port_region"], rel="ARRIVES_AT", days=float(days))

lanes_by_id = lanes.set_index("lane_id")
for ci, (_, c) in enumerate(carriers.iterrows()):
    G.add_node(c["carrier_id"], kind="carrier", name=c["carrier_name"],
               on_time_rate=c["on_time_rate"], base_rate=c["base_rate_usd_per_km_ton"])
    served = lanes.sample(n=6, random_state=1000 + ci)["lane_id"].tolist()
    for lid in served:
        lane = lanes_by_id.loc[lid]
        distance = lane["distance_km"] if not pd.isna(lane["distance_km"]) else 900.0
        days = lane["avg_transit_days"] if not pd.isna(lane["avg_transit_days"]) else 3.0
        G.add_edge(c["carrier_id"], lid, rel="SERVES",
                   cost=round(float(c["base_rate_usd_per_km_ton"]) * float(distance), 2),
                   days=float(days))

print("nodes:", G.number_of_nodes(), "| edges:", G.number_of_edges())
print("kinds:", {k: sum(1 for n, d in G.nodes(data=True) if d.get("kind") == k)
                for k in ("carrier", "lane", "port")})

## Degree centrality

Which carrier serves the most lanes? Degree centrality measures how connected each carrier is  a quick way to spot the network's hubs.

In [ ]:
carrier_nodes = [n for n, d in G.nodes(data=True) if d.get("kind") == "carrier"]
centrality = nx.degree_centrality(G)
top = sorted(carrier_nodes, key=lambda n: centrality[n], reverse=True)[:5]
for n in top:
    print(f"  {G.nodes[n]['name']:22} degree={G.degree(n):3}  centrality={centrality[n]:.3f}")

## Cheapest route to a port

"Cheapest route to Long Beach" is a shortest-path-with-weight query: for each lane arriving at the port, find the cheapest incoming `SERVES` edge. Expressible in one traversal because the relationship is stored explicitly.

In [ ]:
def cheapest_to_port(port):
    best_cost, best_carrier, best_lane = float("inf"), None, None
    for lane in G.predecessors(port):
        if G.nodes[lane].get("kind") != "lane":
            continue
        for carrier in G.predecessors(lane):
            if G.nodes[carrier].get("kind") == "carrier":
                c = G[carrier][lane]["cost"]
                if c < best_cost:
                    best_cost, best_carrier, best_lane = c, carrier, lane
    return best_carrier, best_lane, best_cost

target_port = lanes["port_region"].mode()[0]
carrier, lane, cost = cheapest_to_port(target_port)
print(f"cheapest route to {target_port}:")
print(f"  carrier={G.nodes[carrier]['name']}  lane={lane}  cost=${cost}")

## Shortest path (fewest hops) and a constrained query

`nx.shortest_path` walks the graph. The multi-hop strength shows in the *constrained* version: cheapest route that avoids a given port  a filter on the traversal that a vector search can only approximate.

In [ ]:
# fewest-hops path from a specific carrier to a port it reaches
src = top[0]
reachable = [p for p in ports if nx.has_path(G, src, p)]
if reachable:
    path = nx.shortest_path(G, src, reachable[0])
    print(f"shortest path from {G.nodes[src]['name']} to {reachable[0]} (hops):")
    print("  -> ".join(path))

# constrained: cheapest SERVES cost to target_port, excluding a given lane origin
excluded = "Houston"
cands = [(G[u][v]["cost"], G.nodes[u]["name"], v)
         for u, v in G.edges() if G.nodes[v].get("kind") == "lane" and v in G.predecessors(target_port)
         and G.nodes[v].get("origin") != excluded]
if cands:
    cmin = min(cands)
    print(f"cheapest route to {target_port} avoiding origin '{excluded}': ${cmin[0]} via {cmin[1]} -> {cmin[2]}")

## Optional: the same graph in Neo4j / Cypher

```
# ⚠️ REQUIRES Neo4j (AuraDB free or Docker)
```

The Cypher below expresses the *same* graph and route query declaratively. The notebook prints it and, if a driver + credentials are present, runs it; otherwise it completes without Neo4j.

In [ ]:
cypher_setup = """
CREATE (c:Carrier {name: "Atlas Freight", base_rate: 1.35})
CREATE (la:Lane {id: "L001", cost: 4200, days: 12})
CREATE (p:Port {name: "Long Beach", code: "LGB"})
CREATE (c)-[:SERVES {cost: 4200, days: 12}]->(la)
CREATE (la)-[:ARRIVES_AT]->(p)
"""

cypher_route = """
// Which carriers reach a given port, and at what cost?
MATCH (c:Carrier)-[s:SERVES]->(la:Lane)-[:ARRIVES_AT]->(p:Port {code: "LGB"})
RETURN c.name AS carrier, s.cost AS cost, s.days AS days
ORDER BY s.cost
"""

print("=== Cypher (paste into AuraDB/Docker console) ===\n" + cypher_setup + cypher_route)

import os
ran = False
try:
    from neo4j import GraphDatabase
    uri = os.environ.get("NEO4J_URI")
    pwd = os.environ.get("NEO4J_PASSWORD")
    if uri and pwd:
        driver = GraphDatabase.driver(uri, auth=(os.environ.get("NEO4J_USER", "neo4j"), pwd))
        with driver.session() as s:
            s.run("RETURN 1")  # connectivity check
            result = s.run(cypher_route)
            for rec in result:
                print("  neo4j:", rec)
        driver.close()
        ran = True
except ImportError:
    print("⚠️ neo4j driver not installed (pip install neo4j). Paste the Cypher above into any Neo4j console.")
except Exception as e:
    print("⚠️ Neo4j not reachable:", type(e).__name__, "-", e)
if not ran and not os.environ.get("NEO4J_URI"):
    print("⚠️ Neo4j not configured (set NEO4J_URI + NEO4J_PASSWORD). The graph lab above already completed.")

In [ ]:
# Week 7 · Notebook 2 headline metric: graph size + cheapest route cost to the top port.
print("WEEK7_NB2_GRAPH_EDGES:", G.number_of_edges())
print("WEEK7_NB2_CHEAPEST_ROUTE_USD:", round(cost, 2))